# Held-Out Himalayan Flood Event Evaluation

**Person 2 (AI & Detection) — Triage Disaster Intelligence System**

## 1. Objective and Evaluation Protocol
This notebook defines the independent validation harness for evaluating the multi-channel U-Net and fused detection pipeline on geographically distinct Himalayan disaster events (e.g., Bhote Koshi / Trishuli / Chamoli corridors).

### Scientific Integrity Standards
- **Event-Level Isolation**: No patches from the same satellite acquisition or disaster event may exist in both training and test sets.
- **No Synthetic Masquerading**: Synthetic or augmented patches must NEVER be claimed as genuine Himalayan field observations.
- **No Label Fabrication**: If verified ground-truth labels for an unseen Himalayan event are unavailable, the benchmark evaluation must strictly report **`NOT RUN`**.
- **Isolation of Reference Maps**: EMSR927 or emergency activation delineation layers must NEVER be used as model input features; they are strictly held out as post-hoc reference masks.
- **Zero-Division Safety**: Precision, Recall, IoU, and F1 calculations must handle degenerate zero-denominator cases without crashing or reporting artificial infinities.

In [ ]:
# ─── Environment & Imports ───────────────────────────────────────────────────
import sys, os
from pathlib import Path
import json
import numpy as np

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

from p2_ai.detect.unet import predict_unet, compute_segmentation_metrics
from p2_ai.detect.fuse import fuse_detections, CLASS_BACKGROUND, CLASS_WATER, CLASS_DEBRIS
from p2_ai.detect.sar_change import detect_water_sar
from p2_ai.detect.coherence import detect_debris_amplitude_fallback

print("Loaded detection and evaluation modules successfully.")

## 2. Evaluation Metrics Definition

We compute standard geospatial segmentation metrics over valid pixels:
$$\text{IoU} = \frac{TP}{TP + FP + FN}$$
$$\text{Precision} = \frac{TP}{TP + FP}, \quad \text{Recall} = \frac{TP}{TP + FN}$$
$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Along with 2x2 confusion matrix (TP, FP, FN, TN), pixel support, and valid-pixel counts.

In [ ]:
# ─── Metrics Computation Engine ──────────────────────────────────────────────
def evaluate_event_scene(pred_binary: np.ndarray, gt_binary: np.ndarray, valid_mask: np.ndarray = None) -> dict:
    """Compute comprehensive segmentation metrics with strict denominator safety."""
    if valid_mask is not None:
        p = pred_binary[valid_mask].astype(bool)
        g = gt_binary[valid_mask].astype(bool)
    else:
        p = pred_binary.ravel().astype(bool)
        g = gt_binary.ravel().astype(bool)

    tp = int(np.count_nonzero(p & g))
    fp = int(np.count_nonzero(p & (~g)))
    fn = int(np.count_nonzero((~p) & g))
    tn = int(np.count_nonzero((~p) & (~g)))

    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
    recall = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    f1 = float(2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    iou = float(tp / (tp + fp + fn)) if (tp + fp + fn) > 0 else 0.0

    return {
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": {
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn
        },
        "pixel_support_positive": tp + fn,
        "pixel_support_total": tp + fp + fn + tn,
        "valid_pixel_count": int(len(p))
    }

# Verify metrics function on test vectors
t_pred = np.array([1, 1, 0, 0], dtype=bool)
t_gt   = np.array([1, 0, 1, 0], dtype=bool)
metrics_check = evaluate_event_scene(t_pred, t_gt)
print("Metrics engine verification:", metrics_check)

## 3. Dataset Discovery & Ground Truth Check

We inspect whether independent Himalayan test rasters and corresponding Copernicus EMS / ground validation shapefiles or rasters exist in local storage.

In [ ]:
# ─── Dataset Discovery ───────────────────────────────────────────────────────
HIMALAYAN_DATA_DIR = Path("../data/raw/himalayan_eval")
EVENT_MANIFEST = Path("../data/raw/himalayan_eval/manifest.json")

has_himalayan_data = HIMALAYAN_DATA_DIR.exists() and EVENT_MANIFEST.exists()

print(f"Target Himalayan directory: {HIMALAYAN_DATA_DIR}")
print(f"Data presence: {'FOUND' if has_himalayan_data else 'NOT FOUND'}")

## 4. Benchmark Execution

In accordance with our authoritative scientific contract and anti-fabrication policy:
- If no independent labeled Himalayan scene is present, the benchmark evaluation is reported as **`NOT RUN`**.
- Below we produce the formal evaluation status record.

In [ ]:
# ─── Benchmark Status Generation ─────────────────────────────────────────────
eval_report = {
    "benchmark_name": "Held-Out Himalayan Glacier-Collapse / Flood Evaluation",
    "target_geographic_region": "Bhote Koshi / Trishuli Basin, Nepal / High Mountain Asia",
    "status": "NOT RUN",
    "reason": "Independently georeferenced, ground-truth-validated Himalayan event rasters not present in local storage.",
    "anti_fabrication_policy_compliance": True,
    "pipeline_harness_status": "READY",
    "required_input_rasters": [
        "pre_event_vv_db.tif",
        "pre_event_vh_db.tif",
        "post_event_vv_db.tif",
        "post_event_vh_db.tif",
        "dem_slope_deg.tif",
        "ground_truth_water_mask.tif"
    ],
    "expected_channel_dimensions": "(H, W) identical grid alignment and EPSG CRS",
    "quantitative_metrics": None,
    "failure_mode_analysis": {
        "radar_shadow_false_positives": "Steep mountain backslopes create radar shadow (-24 to -28 dB) resembling specular open water. Mitigated by Phase 3 shadow mask gating.",
        "narrow_valley_headwaters": "Narrow mountain river channels (< 15m) undergo sub-pixel spatial averaging with dry boulder banks in 10m Sentinel-1 and 30m DEM.",
        "saturated_debris_mud": "Post-event saturated debris and wet sediment exhibit significant backscatter drops, requiring multi-sensor coherence and texture analysis to distinguish from standing water."
    }
}

# Output status report
report_path = Path("himalayan_eval_report.json")
with open(report_path, "w") as f:
    json.dump(eval_report, f, indent=2)

print("=== HELD-OUT EVALUATION STATUS ===")
print(f"Status:  {eval_report['status']}")
print(f"Reason:  {eval_report['reason']}")
print(f"Saved formal evaluation record to: {report_path.resolve()}")

## 5. Harness Smoke-Test on Synthetic Orographic Terrain

To guarantee that the evaluation harness and inference pipeline execute without error when rasters are provided, we run an explicit **HARNESS SMOKE TEST** on synthetic mountainous terrain.

*(Notice: These synthetic numbers are solely to verify execution mechanics and must never be reported as real-world Himalayan performance.)*

In [ ]:
# ─── Harness Execution Verification (Smoke Test Only) ────────────────────────
shape = (128, 128)
rng = np.random.default_rng(2026)

# Mountain flanks backscatter
pre_vv = rng.normal(-11.5, 2.0, size=shape).astype(np.float32)
pre_vh = rng.normal(-18.0, 2.5, size=shape).astype(np.float32)
post_vv = pre_vv.copy()
post_vh = pre_vh.copy()
gt_flood = np.zeros(shape, dtype=bool)

# Meandering valley floor corridor
y, x = np.ogrid[:shape[0], :shape[1]]
river_channel = np.abs(x - (shape[1]//2 + np.sin(y/12.0)*15)) < 6
post_vv[river_channel] = rng.normal(-22.0, 1.2, size=np.count_nonzero(river_channel))
post_vh[river_channel] = rng.normal(-27.0, 1.5, size=np.count_nonzero(river_channel))
gt_flood[river_channel] = True

# Slope: steep mountain walls (>25 deg), flat valley floor (<5 deg)
slope = np.abs(x - (shape[1]//2)) * 0.8
slope[river_channel] = 2.5

# Execute full U-Net inference pipeline
result = predict_unet(pre_vv, pre_vh, post_vv, post_vh, slope, threshold=0.50)
smoke_metrics = evaluate_event_scene(result.binary_mask, gt_flood)

print("=== HARNESS SMOKE-TEST METRICS (SYNTHETIC VERIFICATION ONLY) ===")
print(f"Backend:       {result.inference_backend}")
print(f"Smoke IoU:     {smoke_metrics['iou']:.4f}")
print(f"Smoke F1:      {smoke_metrics['f1_score']:.4f}")
print(f"Smoke Prec:    {smoke_metrics['precision']:.4f}")
print(f"Smoke Recall:  {smoke_metrics['recall']:.4f}")
print(f"Confusion:     {smoke_metrics['confusion_matrix']}")
print("Harness execution verified successfully.")

## 6. Documented Physical Failure Modes & Scientific Limitations

1. **Radar Shadow & Layover**:
   - In deep Himalayan gorges, steep relief causes radar layover on foreslopes and radar shadow on backslopes.
   - Radar shadow returns minimal backscatter (-24 to -28 dB), which directly mimics calm water.
   - **Mitigation Requirement**: Must be gated using Person 3's terrain ray-tracing shadow mask (`src/terrain/shadow_mask.py`).

2. **Narrow Channel Spatial Resolution Limits**:
   - Upper tributaries are frequently 5–15 meters wide. Sentinel-1 GRD at 10m pixel spacing causes substantial mixed-pixel contamination with rocky shorelines.
   - Result: Observed recall is structurally lower in narrow V-shaped headwaters.

3. **Wet Silt / Mud / Debris Field Confusion**:
   - Freshly deposited post-flood silt is smooth and saturated, causing specular-like forward scattering that can be mistaken for standing flood water by amplitude alone.
   - **Mitigation**: Dual-polarization VV/VH ratioing combined with coherence loss analysis.